# Bronze → Silver: Chamados_Hora (Limpeza, Tipagem e Correção de Datas)

A primeira vista, é possível notar algumas inconsistências nos dados, como:
- **Horários:** as 3 colunas de data vêm como texto (`dd/MM/yyyy às HH:mm:ss`);
- **Aberturas:** 198 chamados abertos em 2027, mas com início e finalização em 2025/2026;
- **Inícios:** 3.414 chamados sem hora de início;
- **Finalização:** 1 chamado com a finalização antes do início.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType

catalog = "vcredit"
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

print(f"Lendo da Bronze: {bronze_schema}")
print(f"Gravando na Silver: {silver_schema}")

---
## Regras que dá pra ajustar

Valores usados nas regras abaixo: período dos dados, limites de outlier e se o ano da abertura é corrigido ou não.

In [0]:
PERIODO_INICIO = "2025-01-01 00:00:00"
PERIODO_FIM = "2026-07-31 23:59:59"

CORRIGIR_ANO_ABERTURA = True  # True = corrige o ano da abertura
LIMIAR_DIAS_ANO_DIVERGENTE = 180  # abertura 180+ dias depois do início/fim = erro de ano
MAX_DURACAO_CORRECAO_SEG = 86400  # depois de corrigir, o chamado dura até 24h

MAX_ESPERA_SEG = 3600  # espera acima de 1h = outlier (só marca)
MAX_ATENDIMENTO_SEG = 7200  # atendimento acima de 2h = outlier (só marca)

FORMATOS_DATA = [
    "d/M/yyyy H:mm:ss",
    "d/M/yyyy H:mm",
    "yyyy-M-d H:mm:ss",
    "yyyy-M-d H:mm",
    "yyyy-MM-dd'T'HH:mm:ss",
]

---
## 1. Leitura da Bronze (`bronze.tb_chamados_hora`)

O que está sendo feito:
- Lê a Bronze do grupo (tudo em texto);
- Tira os espaços e transforma texto vazio em `NULL`.

In [0]:
df_bronze = spark.table(f"{bronze_schema}.tb_chamados_hora")
total_bronze = df_bronze.count()

def limpa_texto(coluna):
    texto = F.trim(F.col(coluna))
    return F.when(texto == "", F.lit(None)).otherwise(texto)

df_base = df_bronze.select(
    limpa_texto("ID_Chamado").alias("id_chamado_raw"),
    limpa_texto("ID_Cliente").alias("id_cliente_raw"),
    limpa_texto("Hora_Abertura_Chamado").alias("abertura_raw"),
    limpa_texto("Hora_Inicio_Atendimento").alias("inicio_raw"),
    limpa_texto("Hora_Finalizacao_Atendimento").alias("fim_raw"),
    F.col("ingestion_datetime")
)

print(f"Linhas na Bronze: {total_bronze}")

---
## 2. IDs inválidos e duplicados

O que está sendo feito:
- **`id_chamado`** vazio ou que não é número vai pra quarentena;
- **`id_chamado`** repetido: fica a linha mais completa (empate: a mais recente e depois o `hash_linha`). As outras vão pra quarentena.

In [0]:
colunas_raw = ["id_chamado_raw", "id_cliente_raw", "abertura_raw", "inicio_raw", "fim_raw"]

df_base = (
    df_base
    .withColumn("id_chamado_txt", F.regexp_replace(F.col("id_chamado_raw"), r"\.0+$", ""))
    .withColumn("id_valido", F.coalesce(F.col("id_chamado_txt").rlike(r"^\d{1,9}$"), F.lit(False)))
    .withColumn("hash_linha", F.sha2(F.concat_ws("|", *[F.coalesce(F.col(c), F.lit("")) for c in colunas_raw]), 256))
)

df_invalidos = df_base.filter(~F.col("id_valido")).withColumn("motivo", F.lit("ID_CHAMADO_AUSENTE_OU_INVALIDO"))

campos_vazios = sum([F.when(F.col(c).isNull(), 1).otherwise(0) for c in colunas_raw[1:]])

win_chamado = Window.partitionBy("id_chamado_txt").orderBy(
    campos_vazios.asc(), F.col("ingestion_datetime").desc(), F.col("hash_linha").asc()
)
win_contagem = Window.partitionBy("id_chamado_txt")

df_validos = (
    df_base.filter(F.col("id_valido"))
    .withColumn("rn", F.row_number().over(win_chamado))
    .withColumn("qtd_linhas_bronze", F.count("*").over(win_contagem))
)

df_duplicados = df_validos.filter(F.col("rn") > 1).withColumn("motivo", F.lit("ID_CHAMADO_DUPLICADO_DESCARTADO"))
df_unicos = df_validos.filter(F.col("rn") == 1).drop("rn")

def formata_quarentena(df):
    return df.select(
        F.col("id_chamado_raw").alias("id_chamado"),
        F.col("id_cliente_raw").alias("id_cliente"),
        F.col("abertura_raw").alias("hora_abertura_chamado"),
        F.col("inicio_raw").alias("hora_inicio_atendimento"),
        F.col("fim_raw").alias("hora_finalizacao_atendimento"),
        F.col("motivo"),
        F.col("ingestion_datetime")
    )

df_quarentena = formata_quarentena(df_invalidos).unionByName(formata_quarentena(df_duplicados))

---
## 3. Tipagem

O que está sendo feito:
- **`id_chamado`**: string pra inteiro (`IntegerType`);
- **`id_cliente`**: continua texto;
- **Horários**: viram `TimestampType`.

In [0]:
def converte_data(coluna):
    texto = F.regexp_replace(
        F.col(coluna),
        r"^(\d{1,2}/\d{1,2}/\d{4})\D+(\d{1,2}:\d{2}(:\d{2})?)$",
        "$1 $2"
    )
    return F.coalesce(*[F.try_to_timestamp(texto, F.lit(formato)) for formato in FORMATOS_DATA])

cliente_limpo = F.regexp_replace(F.regexp_replace(F.col("id_cliente_raw"), r"\.0+$", ""), r"[\.\-\s/]", "")

df_tipado = df_unicos.select(
    F.col("id_chamado_txt").cast(IntegerType()).alias("id_chamado"),
    F.when(cliente_limpo.rlike(r"^\d+$"), cliente_limpo).alias("id_cliente"),
    converte_data("abertura_raw").alias("hora_abertura_original"),
    converte_data("inicio_raw").alias("hora_inicio_atendimento"),
    converte_data("fim_raw").alias("hora_finalizacao_atendimento"),
    F.col("abertura_raw").isNotNull().alias("veio_abertura"),
    F.col("inicio_raw").isNotNull().alias("veio_inicio"),
    F.col("fim_raw").isNotNull().alias("veio_fim"),
    F.col("qtd_linhas_bronze"),
    F.col("ingestion_datetime")
)

---
## 4. Correção do ano da abertura

O que está sendo feito:
- É o **único** horário alterado: a abertura de 2027 passa pro ano do início (ou da finalização);
- Só corrige se a abertura vier 180 dias ou mais depois do início/fim e o chamado resultante durar até 24h;
- O valor original fica em **`hora_abertura_original`**.

In [0]:
df_ano = (
    df_tipado
    .withColumn("ref_ts", F.coalesce("hora_inicio_atendimento", "hora_finalizacao_atendimento"))
    .withColumn("fim_ref", F.coalesce("hora_finalizacao_atendimento", "hora_inicio_atendimento"))
    .withColumn("dif_anos", F.year("ref_ts") - F.year("hora_abertura_original"))
    .withColumn("abertura_try1", F.expr("hora_abertura_original + make_interval(dif_anos, 0, 0, 0, 0, 0, 0)"))
    .withColumn(
        "abertura_ajustada",
        F.when(
            F.col("abertura_try1") > F.col("ref_ts"),
            F.expr("hora_abertura_original + make_interval(dif_anos - 1, 0, 0, 0, 0, 0, 0)")
        ).otherwise(F.col("abertura_try1"))
    )
    .withColumn(
        "corr_ano_abertura",
        F.lit(CORRIGIR_ANO_ABERTURA) & F.coalesce(
            (F.datediff("hora_abertura_original", "ref_ts") >= LIMIAR_DIAS_ANO_DIVERGENTE)
            & (F.col("fim_ref").cast("long") - F.col("abertura_ajustada").cast("long")).between(0, MAX_DURACAO_CORRECAO_SEG),
            F.lit(False)
        )
    )
    .withColumn(
        "hora_abertura_chamado",
        F.when(F.col("corr_ano_abertura"), F.col("abertura_ajustada")).otherwise(F.col("hora_abertura_original"))
    )
)

---
## 5. Tempos e situação do atendimento

O que está sendo feito:
- **`espera_seg`** (início − abertura), **`atendimento_seg`** (finalização − início) e **`duracao_total_seg`**;
- **`situacao_atendimento`**: `ATENDIDO`, `NAO_ATENDIDO` (sem início, provável abandono), `SEM_FINALIZACAO` ou `SEM_HORARIOS`;
- **`espera_valida` / `atendimento_valido`**: tempo maior ou igual a zero (usar nas médias).

In [0]:
df_calc = (
    df_ano
    .withColumn("espera_seg", F.col("hora_inicio_atendimento").cast("long") - F.col("hora_abertura_chamado").cast("long"))
    .withColumn("atendimento_seg", F.col("hora_finalizacao_atendimento").cast("long") - F.col("hora_inicio_atendimento").cast("long"))
    .withColumn("duracao_total_seg", F.col("hora_finalizacao_atendimento").cast("long") - F.col("hora_abertura_chamado").cast("long"))
    .withColumn(
        "situacao_atendimento",
        F.when(F.col("hora_inicio_atendimento").isNotNull() & F.col("hora_finalizacao_atendimento").isNotNull(), "ATENDIDO")
         .when(F.col("hora_inicio_atendimento").isNull() & F.col("hora_finalizacao_atendimento").isNotNull(), "NAO_ATENDIDO")
         .when(F.col("hora_inicio_atendimento").isNotNull(), "SEM_FINALIZACAO")
         .otherwise("SEM_HORARIOS")
    )
    .withColumn(
        "tempo_ate_desistencia_seg",
        F.when((F.col("situacao_atendimento") == "NAO_ATENDIDO") & (F.col("duracao_total_seg") >= 0), F.col("duracao_total_seg"))
    )
    .withColumn("espera_valida", F.coalesce(F.col("espera_seg") >= 0, F.lit(False)))
    .withColumn("atendimento_valido", F.coalesce(F.col("atendimento_seg") >= 0, F.lit(False)))
    .withColumn("data_abertura", F.to_date("hora_abertura_chamado"))
    .withColumn("hora_abertura", F.hour("hora_abertura_chamado"))
    .withColumn("ano_mes", F.date_format("hora_abertura_chamado", "yyyy-MM"))
)

---
## 6. Marcações de qualidade

O que está sendo feito:
- Cada problema vira uma coluna `dq_...` (`true`/`false`), nada é apagado;
- Finalização antes do início **não é alterada**, só marcada;
- **`dq_status`**: `INCONSISTENTE`, `INCOMPLETO`, `CORRIGIDO` ou `OK` (vale o primeiro que se aplicar).

In [0]:
df_flags = (
    df_calc
    .withColumn("dq_id_duplicado", F.col("qtd_linhas_bronze") > 1)
    .withColumn("dq_id_cliente_nulo", F.col("id_cliente").isNull())
    .withColumn("dq_abertura_nula", F.col("hora_abertura_chamado").isNull())
    .withColumn("dq_inicio_nulo", F.col("hora_inicio_atendimento").isNull())
    .withColumn("dq_fim_nulo", F.col("hora_finalizacao_atendimento").isNull())
    .withColumn(
        "dq_ts_ilegivel",
        (F.col("veio_abertura") & F.col("hora_abertura_original").isNull())
        | (F.col("veio_inicio") & F.col("hora_inicio_atendimento").isNull())
        | (F.col("veio_fim") & F.col("hora_finalizacao_atendimento").isNull())
    )
    .withColumn("dq_inicio_antes_abertura", F.coalesce(F.col("espera_seg") < 0, F.lit(False)))
    .withColumn("dq_fim_antes_inicio", F.coalesce(F.col("atendimento_seg") < 0, F.lit(False)))
    .withColumn("dq_fim_antes_abertura", F.coalesce(F.col("duracao_total_seg") < 0, F.lit(False)))
    .withColumn(
        "dq_fora_periodo",
        F.coalesce(
            (F.col("hora_abertura_chamado") < F.lit(PERIODO_INICIO).cast("timestamp"))
            | (F.col("hora_abertura_chamado") > F.lit(PERIODO_FIM).cast("timestamp")),
            F.lit(False)
        )
    )
    .withColumn("dq_espera_outlier", F.coalesce(F.col("espera_seg") > MAX_ESPERA_SEG, F.lit(False)))
    .withColumn("dq_atendimento_outlier", F.coalesce(F.col("atendimento_seg") > MAX_ATENDIMENTO_SEG, F.lit(False)))
    .withColumn(
        "dq_status",
        F.when(
            F.col("dq_ts_ilegivel") | F.col("dq_abertura_nula") | F.col("dq_inicio_antes_abertura")
            | F.col("dq_fim_antes_inicio") | F.col("dq_fim_antes_abertura") | F.col("dq_fora_periodo"),
            "INCONSISTENTE"
        )
         .when(F.col("dq_inicio_nulo") | F.col("dq_fim_nulo") | F.col("dq_id_cliente_nulo"), "INCOMPLETO")
         .when(F.col("corr_ano_abertura"), "CORRIGIDO")
         .otherwise("OK")
    )
)

---
## 7. Gravação

Grava `silver.tb_chamados_hora` e a quarentena em Delta (`overwrite`), igual às outras tabelas.

In [0]:
df_silver_chamados_hora = df_flags.select(
    "id_chamado",
    "id_cliente",
    "hora_abertura_chamado",
    "hora_inicio_atendimento",
    "hora_finalizacao_atendimento",
    "hora_abertura_original",
    "espera_seg",
    "atendimento_seg",
    "duracao_total_seg",
    "tempo_ate_desistencia_seg",
    "situacao_atendimento",
    "espera_valida",
    "atendimento_valido",
    "data_abertura",
    "hora_abertura",
    "ano_mes",
    "corr_ano_abertura",
    "dq_id_duplicado",
    "dq_id_cliente_nulo",
    "dq_abertura_nula",
    "dq_inicio_nulo",
    "dq_fim_nulo",
    "dq_ts_ilegivel",
    "dq_inicio_antes_abertura",
    "dq_fim_antes_inicio",
    "dq_fim_antes_abertura",
    "dq_fora_periodo",
    "dq_espera_outlier",
    "dq_atendimento_outlier",
    "dq_status",
    "ingestion_datetime"
)

(
    df_silver_chamados_hora.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(f"{silver_schema}.tb_chamados_hora")
)

(
    df_quarentena.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(f"{silver_schema}.tb_chamados_hora_quarentena")
)

print(f"Tabela {silver_schema}.tb_chamados_hora salva com sucesso!")
print(f"Tabela {silver_schema}.tb_chamados_hora_quarentena salva com sucesso!")

---
Assim, a tabela `silver.tb_chamados_hora` está pronta, tipada e com os problemas tratados ou marcados.